### Import und Datenaufbereitung

In [63]:
# Import
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [64]:
# Excel Datei laden
df = pd.read_excel("../data/CTG.xls", sheet_name="Raw Data")

# erste Übersicht
print(df.head())
print(df.shape)
print(df.info())

       FileName       Date      SegFile      b       e    LBE     LB   AC  \
0           NaN        NaT          NaN    NaN     NaN    NaN    NaN  NaN   
1  Variab10.txt 1996-12-01  CTG0001.txt  240.0   357.0  120.0  120.0  0.0   
2    Fmcs_1.txt 1996-05-03  CTG0002.txt    5.0   632.0  132.0  132.0  4.0   
3    Fmcs_1.txt 1996-05-03  CTG0003.txt  177.0   779.0  133.0  133.0  2.0   
4    Fmcs_1.txt 1996-05-03  CTG0004.txt  411.0  1192.0  134.0  134.0  2.0   

    FM   UC  ...    C    D    E   AD   DE   LD   FS  SUSP  CLASS  NSP  
0  NaN  NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN   NaN    NaN  NaN  
1  0.0  0.0  ...  0.0  0.0  0.0  0.0  0.0  0.0  1.0   0.0    9.0  2.0  
2  0.0  4.0  ...  0.0  0.0  0.0  1.0  0.0  0.0  0.0   0.0    6.0  1.0  
3  0.0  5.0  ...  0.0  0.0  0.0  1.0  0.0  0.0  0.0   0.0    6.0  1.0  
4  0.0  6.0  ...  0.0  0.0  0.0  1.0  0.0  0.0  0.0   0.0    6.0  1.0  

[5 rows x 40 columns]
(2130, 40)
<class 'pandas.DataFrame'>
RangeIndex: 2130 entries, 0 to 2129
Data col

In [65]:
# Zeilen mit fehlenden Werten prüfen, da die meisten Spalten nur 2126 Non-Null haben
df[df.isnull().any(axis=1)]

,FileName,Date,SegFile,b,e,LBE,LB,AC,FM,UC,...,C,D,E,AD,DE,LD,FS,SUSP,CLASS,NSP
0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2127,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2128,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2129,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,564.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
# Zeilen mit fehlenden Werten enthalten keine Zielvariable, daher löschen
df = df.dropna()
print(df.isnull().sum())
print(df.shape)

FileName    0
Date        0
SegFile     0
b           0
e           0
LBE         0
LB          0
AC          0
FM          0
UC          0
ASTV        0
MSTV        0
ALTV        0
MLTV        0
DL          0
DS          0
DP          0
DR          0
Width       0
Min         0
Max         0
Nmax        0
Nzeros      0
Mode        0
Mean        0
Median      0
Variance    0
Tendency    0
A           0
B           0
C           0
D           0
E           0
AD          0
DE          0
LD          0
FS          0
SUSP        0
CLASS       0
NSP         0
dtype: int64
(2126, 40)


### Feature-Auswahl

Zunächst wurden Variablen entfernt, die keine relevanten Informationen für die Klassifikation liefern oder bereits eine abgeleitete Bewertung des CTG-Signals darstellen.

- **FileName, Date, SegFile, b, e** enthalten lediglich Metadaten der Untersuchung (Dateiname, Zeitpunkt bzw. Start- und Endinstanz der Messung) und tragen keine inhaltliche Information zur Klassifikation des fetalen Zustands bei.

- **A, B, C, D, E, AD, DE, LD, FS, SUSP, CLASS** beschreiben bereits klassifizierte Muster des CTG-Signals oder diagnostische Bewertungen. Da sie bereits interpretierte Informationen enthalten, könnten sie zu einer Verzerrung des Modells führen (Target Leakage). Das Modell soll den fetalen Zustand direkt aus den physiologischen Merkmalen ableiten. Unter https://archive.ics.uci.edu/ml/datasets/Cardiotocography wird **CLASS** zudem als Target Variable beschrieben.

In [67]:
# Unnötige Spalten entfernen
drop_cols = [
    "FileName", "Date", "SegFile", "b", "e", "A", "B", "C", "D", "E", "AD", "DE", "LD", "FS", "SUSP", "CLASS"
]
df = df.drop(columns=drop_cols)

# Features und Zielvariable "NSP" trennen
X = df.drop(columns=["NSP"])
y = df["NSP"]

### Wahl des Algorithmus
Für die Klassifikation der Zielvariable NSP wird ein Random-Forest-Classifier gewählt.
Der Algorithmus eignet sich besonders gut für strukturierte tabellarische Daten mit mehreren numerischen Merkmalen, wie sie im CTG-Datensatz vorliegen. Random Forest ist bekannt dafür komplexe und nichtlineare Zusammenhänge zwischen z. B. medizinischen Messwerten zu erkennen.

Ein weiterer Vorteil ist die Robustheit gegenüber Ausreißern sowie die Tatsache, dass keine Feature-Skalierung notwendig ist. Durch das Ensemble-Prinzip wird außerdem die Gefahr von Overfitting reduziert, da viele Entscheidungsbäume auf zufälligen Teilmengen der Daten trainiert werden.

In [68]:
# Train-Test-Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y # wichtig, um die Klassenverteilung im Train- und Testset beizubehalten
)

# Random Forest Modell
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

# Training
rf_model.fit(X_train, y_train)

# Vorhersage
y_pred = rf_model.predict(X_test)

# Accuracy berechnen
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.931924882629108


### Hyperparameter-Optimierung mit GridSearchCV

Nach dem Training des Basismodells wurde eine Hyperparameter-Optimierung mit `GridSearchCV` durchgeführt.  
Dabei wurden verschiedene Kombinationen der Parameter `n_estimators`, `max_depth`, `min_samples_split` und `min_samples_leaf` getestet.

Das Basismodell erreichte bereits eine Accuracy von **93,19 %**.

Die GridSearch ergab folgende optimale Parameter:

- `n_estimators = 200`
- `max_depth = None`
- `min_samples_split = 2`
- `min_samples_leaf = 1`

Die Kreuzvalidierung erreichte eine durchschnittliche Accuracy von **94,35 %**.  
Das zeigt: Das Random-Forest-Modell liefert bereits mit den Standardparametern sehr gute Ergebnisse, aber konnte durch eine Erhöhung der Anzahl an Entscheidungsbäumen verbessert werden.

Die Optimierung bestätigt die Eignung des Random-Forest-Algorithmus für diesen Datensatz, da eine hohe Klassifikationsgenauigkeit mit relativ geringem Parameter-Tuning erreicht wurde.

In [69]:
# Parameter Grid für GridSearchCV
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Training mit GridSearch
grid_search.fit(X_train, y_train)

# Bestes Modell
best_rf = grid_search.best_estimator_

# Vorhersage auf Testdaten
y_pred_best = best_rf.predict(X_test)

print("Beste Parameter:", grid_search.best_params_)
print("Beste Cross-Validation Accuracy:", grid_search.best_score_)

Beste Parameter: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Beste Cross-Validation Accuracy: 0.943529411764706
